In [1]:
# ================================================
# NOTEBOOK 2 — DATA PREPROCESSING
# SaaS Churn Prediction Project
# ================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:
# ================================================
# STEP 1 — LOAD DATA
# ================================================

df = pd.read_csv(r'C:\Users\shriv\Documents\churn_prediction\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"✅ Data loaded!")
print(f"   Rows:    {df.shape[0]}")
print(f"   Columns: {df.shape[1]}")
df.head()



✅ Data loaded!
   Rows:    7043
   Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# ================================================
# STEP 2 — CLEAN DATA
# ================================================

# Fix TotalCharges (loaded as text, convert to number)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Check missing values before cleaning
print("Missing values found:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Drop rows with missing values
df.dropna(inplace=True)

print(f"\n✅ Data cleaned!")
print(f"   Rows after cleaning: {df.shape[0]}")
print(f"   Rows removed: {7043 - df.shape[0]}")

Missing values found:
TotalCharges    11
dtype: int64

✅ Data cleaned!
   Rows after cleaning: 7032
   Rows removed: 11


In [4]:
# ================================================
# STEP 3 — ENCODE CATEGORICAL COLUMNS
# ================================================

# Convert Yes/No columns to 1/0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df['gender'] = (df['gender'] == 'Male').astype(int)
df['Partner'] = (df['Partner'] == 'Yes').astype(int)
df['Dependents'] = (df['Dependents'] == 'Yes').astype(int)
df['PhoneService'] = (df['PhoneService'] == 'Yes').astype(int)
df['PaperlessBilling'] = (df['PaperlessBilling'] == 'Yes').astype(int)

# Convert remaining text columns
df = pd.get_dummies(df, columns=[
    'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'
], drop_first=True)

# Drop customer ID
df.drop(columns=['customerID'], inplace=True)

print(f"✅ Encoding done!")
print(f"   Total columns now: {df.shape[1]}")
print(f"   Sample columns: {df.columns.tolist()[:5]}")

✅ Encoding done!
   Total columns now: 31
   Sample columns: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure']


In [5]:
# ================================================
# STEP 4 — SPLIT DATA & BALANCE CLASSES
# ================================================

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print(f"Features: {X.shape[1]}")
print(f"Target distribution before SMOTE:")
print(f"   Stay:  {y.value_counts()[0]} ({y.value_counts(normalize=True)[0]:.1%})")
print(f"   Churn: {y.value_counts()[1]} ({y.value_counts(normalize=True)[1]:.1%})")

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Apply SMOTE to fix class imbalance
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"\n✅ Split and balancing done!")
print(f"   Training rows after SMOTE: {X_train_res.shape[0]}")
print(f"   Testing rows:              {X_test.shape[0]}")
print(f"   Balance after SMOTE:")
print(f"   Stay:  {y_train_res.value_counts()[0]}")
print(f"   Churn: {y_train_res.value_counts()[1]}")

Features: 30
Target distribution before SMOTE:
   Stay:  5163 (73.4%)
   Churn: 1869 (26.6%)

✅ Split and balancing done!
   Training rows after SMOTE: 8260
   Testing rows:              1407
   Balance after SMOTE:
   Stay:  4130
   Churn: 4130


In [6]:
# ================================================
# STEP 5 — SAVE PROCESSED DATA
# ================================================

import os

# Create processed folder if not exists
os.makedirs(r'C:\Users\shriv\Documents\churn_prediction\data\processed', exist_ok=True)

# Save all splits
X_train_res.to_csv(r'C:\Users\shriv\Documents\churn_prediction\data\processed\X_train.csv', index=False)
X_test.to_csv(r'C:\Users\shriv\Documents\churn_prediction\data\processed\X_test.csv', index=False)
y_train_res.to_csv(r'C:\Users\shriv\Documents\churn_prediction\data\processed\y_train.csv', index=False)
y_test.to_csv(r'C:\Users\shriv\Documents\churn_prediction\data\processed\y_test.csv', index=False)

print("✅ Processed data saved!")
print(f"   X_train: {X_train_res.shape}")
print(f"   X_test:  {X_test.shape}")
print(f"   y_train: {y_train_res.shape}")
print(f"   y_test:  {y_test.shape}")
print(f"\n📁 Saved to: data/processed/")

✅ Processed data saved!
   X_train: (8260, 30)
   X_test:  (1407, 30)
   y_train: (8260,)
   y_test:  (1407,)

📁 Saved to: data/processed/
